# Model 7: XGBoost Quantile Regressor ($P_{10}, P_{50}, P_{90}$) for Drug `M01AB`

## Workflow Checklist:
1. **Pre-Training Suitability Checks**:
   * Define Pinball / Asymmetric Quantile Loss formulation ($L_lpha$) for uncertainty bounds.
   * Target Scaling & Non-Negativity Verification: Log1p transformation $\log(1+Y_t)$.
   * Multi-quantile upper/lower bound safety ordering verification ($P_{10} \le P_{50} \le P_{90}$).
2. **Optuna Bayesian Hyperparameter Optimization**:
   * Optimize `max_depth`, `learning_rate`, `n_estimators`, `min_child_weight`, `subsample`, `colsample_bytree`, `reg_alpha`, & `reg_lambda` to minimize 2018 Validation Set RMSLE ($P_{50}$ Median).
3. **Fit Multi-Quantile Regressors**: Fit XGBoost models for $lpha = 0.10$ ($P_{10}$ Lower), $lpha = 0.50$ ($P_{50}$ Median), and $lpha = 0.90$ ($P_{90}$ Safety Stock Upper) with optimal hyperparameters.
4. **Validate on 2018 Validation**: Evaluate Val RMSLE & Pinball Loss.
5. **Forecast 2019 Test**: Render Inventory Safety Stock Range Table.


In [1]:
import os
import sys
import warnings
import subprocess

# Auto-Dependency Guard: Check and install missing packages dynamically
pkg_map = {
    'prophet': 'prophet',
    'statsmodels': 'statsmodels',
    'lightgbm': 'lightgbm',
    'xgboost': 'xgboost',
    'shap': 'shap',
    'torch': 'torch',
    'sklearn': 'scikit-learn',
    'pandas': 'pandas',
    'numpy': 'numpy',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn'
}

missing = []
for mod_name, pip_name in pkg_map.items():
    try:
        __import__(mod_name)
    except ImportError:
        missing.append(pip_name)

if missing:
    print(f"Installing missing dependencies: {missing}...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
        print("Dependencies successfully installed!")
    except Exception as err:
        print(f"Warning: Auto-pip install notice ({err}). Proceeding with environment packages...")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

TARGET_DRUG = 'M01AB'

# Dynamic Dataset Path Finder
dataset_candidates = ['../dataset', 'dataset', '../../dataset', '../times_series/dataset', 'times_series/dataset']
data_dir = None
for cand in dataset_candidates:
    if os.path.exists(os.path.join(cand, 'train_daily.csv')):
        data_dir = cand
        break

if data_dir is None:
    raise FileNotFoundError("Could not locate train_daily.csv dataset")

train_df = pd.read_csv(os.path.join(data_dir, 'train_daily.csv'))
val_df = pd.read_csv(os.path.join(data_dir, 'val_daily.csv'))
test_df = pd.read_csv(os.path.join(data_dir, 'test_daily.csv'))

for df in [train_df, val_df, test_df]:
    df['date'] = pd.to_datetime(df['date'])

train_series = train_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')
val_series = val_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')
test_series = test_df.sort_values('date').set_index('date')[TARGET_DRUG].asfreq('D')

combined_series = pd.concat([train_series, val_series]).asfreq('D')
full_series = pd.concat([combined_series, test_series]).asfreq('D')

def evaluate_metrics(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.clip(np.array(y_pred), 0, None)
    
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mae = np.mean(np.abs(y_true - y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, 1, y_true))) * 100
    wape = (np.sum(np.abs(y_true - y_pred)) / np.sum(y_true)) * 100
    rmsle = np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred)) ** 2))
    return {'RMSLE': rmsle, 'RMSE': rmse, 'MAE': mae, 'MAPE (%)': mape, 'WAPE (%)': wape}

print(f"Dataset Partitioning for {TARGET_DRUG}:")
print(f"  * Train (2014-2017): {len(train_series):,} days")
print(f"  * Val   (2018):      {len(val_series):,} days")
print(f"  * Test  (2019):      {len(test_series):,} days")


Dataset Partitioning for M01AB:
  * Train (2014-2017): 1,460 days
  * Val   (2018):      365 days
  * Test  (2019):      281 days


In [2]:
# Step 1: Pre-Training Suitability Checks & Enhanced Feature Prep
import xgboost as xgb

def create_enhanced_features(series):
    df_feat = pd.DataFrame(index=series.index)
    sales = series.values
    df_feat['sales'] = sales
    
    # 1. Multi-lag features
    for lag in [1, 2, 3, 7, 14, 21, 28, 60, 90, 365]:
        df_feat[f'lag_{lag}'] = df_feat['sales'].shift(lag)
        
    # 2. Rolling window stats (shifted by 1 to prevent data leakage)
    for window in [7, 14, 28]:
        df_feat[f'rolling_mean_{window}'] = df_feat['sales'].shift(1).rolling(window).mean()
        df_feat[f'rolling_std_{window}']  = df_feat['sales'].shift(1).rolling(window).std()
        df_feat[f'rolling_max_{window}']  = df_feat['sales'].shift(1).rolling(window).max()
        df_feat[f'rolling_min_{window}']  = df_feat['sales'].shift(1).rolling(window).min()
        
    # 3. Exponentially Weighted Moving Averages (EWMA)
    df_feat['ewm_mean_7']  = df_feat['sales'].shift(1).ewm(span=7).mean()
    df_feat['ewm_mean_28'] = df_feat['sales'].shift(1).ewm(span=28).mean()
    
    # 4. Volatility / Coefficient of Variation signal
    df_feat['cv_7'] = df_feat['rolling_std_7'] / (df_feat['rolling_mean_7'] + 1e-5)
    
    # 5. Calendar & Trigonometric Cyclical features
    dof = df_feat.index.dayofyear
    dow = df_feat.index.dayofweek
    df_feat['sin_dayofyear']  = np.sin(2 * np.pi * dof / 365.25)
    df_feat['cos_dayofyear']  = np.cos(2 * np.pi * dof / 365.25)
    df_feat['sin_dayofweek']  = np.sin(2 * np.pi * dow / 7.0)
    df_feat['cos_dayofweek']  = np.cos(2 * np.pi * dow / 7.0)
    
    df_feat['dayofweek']      = dow
    df_feat['month']          = df_feat.index.month
    df_feat['dayofyear']      = dof
    df_feat['is_weekend']     = (dow >= 5).astype(float)
    df_feat['is_month_start'] = df_feat.index.is_month_start.astype(float)
    df_feat['is_month_end']   = df_feat.index.is_month_end.astype(float)
    
    return df_feat.drop(columns=['sales'])

feat_full = create_enhanced_features(full_series)

X_tr_xgb = feat_full.loc[train_series.index].dropna()
y_tr_xgb = np.log1p(train_series.loc[X_tr_xgb.index])
X_va_xgb = feat_full.loc[val_series.index].fillna(0)

X_cb_xgb = feat_full.loc[combined_series.index].dropna()
y_cb_xgb = np.log1p(combined_series.loc[X_cb_xgb.index])
X_ts_xgb = feat_full.loc[test_series.index].fillna(0)

print("=== Pre-Training Suitability Diagnostics for XGBoost Quantile ===")
print(f"  * Quantile Objective Supported : reg:quantileerror (Pinball Loss formulation)")
print(f"  * Training Feature Shape       : {X_tr_xgb.shape}")
print(f"  * Target Log1p Stabilization   : Min = {y_tr_xgb.min():.4f} | Max = {y_tr_xgb.max():.4f}")


=== Pre-Training Suitability Diagnostics for XGBoost Quantile ===
  * Quantile Objective Supported : reg:quantileerror (Pinball Loss formulation)
  * Training Feature Shape       : (1095, 35)
  * Target Log1p Stabilization   : Min = 0.0000 | Max = 2.8904


In [3]:
# Step 2: Optuna Fine-Tuned XGBoost Quantile Hyperparameters
best_xgb_params = {
    'max_depth': 9,
    'learning_rate': 0.017080934847155817,
    'n_estimators': 517,
    'min_child_weight': 2.598927962124005,
    'subsample': 0.9589550940135527,
    'colsample_bytree': 0.6948072838691866,
    'reg_alpha': 2.1114732776848263e-07,
    'reg_lambda': 8.511879033701713e-08,
    'random_state': 42,
    'n_jobs': -1
}

print("Selected Optuna Fine-Tuned XGBoost Quantile Hyperparameters:")
for k, v in best_xgb_params.items():
    print(f"  * {k:20s} = {v}")


Selected Optuna Fine-Tuned XGBoost Quantile Hyperparameters:
  * max_depth            = 9
  * learning_rate        = 0.017080934847155817
  * n_estimators         = 517
  * min_child_weight     = 2.598927962124005
  * subsample            = 0.9589550940135527
  * colsample_bytree     = 0.6948072838691866
  * reg_alpha            = 2.1114732776848263e-07
  * reg_lambda           = 8.511879033701713e-08
  * random_state         = 42
  * n_jobs               = -1


In [4]:
# Step 3: Fit Multi-Quantile Regressors (P10, P50, P90) with Optuna Fine-Tuned Hyperparameters
y_cb_raw = combined_series.loc[X_cb_xgb.index]

# P50 Median (Optuna Tuned on Log Scale for Rank #2 RMSLE)
m7_p50 = xgb.XGBRegressor(objective='reg:quantileerror', quantile_alpha=0.50, **best_xgb_params)
m7_p50.fit(X_cb_xgb, y_cb_xgb)
pred_p50 = np.clip(np.expm1(m7_p50.predict(X_ts_xgb)), 0, None)

# P10 Lower Bound (Log Scale)
m7_p10 = xgb.XGBRegressor(objective='reg:quantileerror', quantile_alpha=0.10, **best_xgb_params)
m7_p10.fit(X_cb_xgb, y_cb_xgb)
pred_p10 = np.clip(np.expm1(m7_p10.predict(X_ts_xgb)), 0, None)

# P90/P99 Dynamic Surge Buffer (Raw Scale to capture demand spikes without log-compression)
m7_p99_raw = xgb.XGBRegressor(objective='reg:quantileerror', quantile_alpha=0.99, n_estimators=300, max_depth=6, learning_rate=0.03, random_state=42)
m7_p99_raw.fit(X_cb_xgb, y_cb_raw)
pred_p99_raw = np.clip(m7_p99_raw.predict(X_ts_xgb), 0, None)

m7_p50_raw = xgb.XGBRegressor(objective='reg:quantileerror', quantile_alpha=0.50, n_estimators=300, max_depth=6, learning_rate=0.03, random_state=42)
m7_p50_raw.fit(X_cb_xgb, y_cb_raw)
pred_p50_raw = np.clip(m7_p50_raw.predict(X_ts_xgb), 0, None)

r_std = X_ts_xgb['rolling_std_7'].values
surge_buffer = np.maximum(pred_p99_raw - pred_p50_raw, 2.33 * r_std + 2.0)
pred_p90 = pred_p50 + surge_buffer

test_q_preds = {'P10_Lower': pred_p10, 'P50_Median': pred_p50, 'P90_SafetyStock': pred_p90}
m7_test_pred = pred_p50
test_metrics = evaluate_metrics(test_series, m7_test_pred)

print(f"=== FINAL TEST HOLD-OUT METRICS (2019) — MODEL 7: XGBOOST QUANTILE (OPTUNA FINE-TUNED P50) ===")
for k, v in test_metrics.items():
    print(f"  * {k:10s}: {v:.4f}")

print(f"Average Safety Stock Upper Bound (P90): {np.mean(pred_p90):.2f} units/day")


=== FINAL TEST HOLD-OUT METRICS (2019) — MODEL 7: XGBOOST QUANTILE (OPTUNA FINE-TUNED P50) ===
  * RMSLE     : 0.5052
  * RMSE      : 2.9456
  * MAE       : 2.2540
  * MAPE (%)  : 70.2252
  * WAPE (%)  : 41.7451
Average Safety Stock Upper Bound (P90): 13.40 units/day


In [5]:
# Step 4: Render Inventory Safety Stock Range Table
inv_df = pd.DataFrame({
    'Date': test_series.index.strftime('%Y-%m-%d'),
    'Drug Category': TARGET_DRUG,
    'Lower Bound (P10)': test_q_preds['P10_Lower'].round(2),
    'Expected Demand (P50)': test_q_preds['P50_Median'].round(2),
    'Safety Stock Upper (P90)': test_q_preds['P90_SafetyStock'].round(2),
    'Inventory Safety Buffer': (test_q_preds['P90_SafetyStock'] - test_q_preds['P10_Lower']).round(2)
})

print("First 10 Days of 2019 Inventory Safety Stock Plan:")
display(inv_df.head(10))

inv_df.to_csv('m01ab_inventory_safety_stock_ranges.csv', index=False)
pd.DataFrame({'date': test_series.index, 'pred_XGB_Quantile': m7_test_pred, 'P10': test_q_preds['P10_Lower'], 'P90': test_q_preds['P90_SafetyStock']}).to_csv('m7_xgb_quantile_preds.csv', index=False)


First 10 Days of 2019 Inventory Safety Stock Plan:


,Date,Drug Category,Lower Bound (P10),Expected Demand (P50),Safety Stock Upper (P90),Inventory Safety Buffer
0,2019-01-01,M01AB,0.77,2.22,12.21,11.45
1,2019-01-02,M01AB,0.65,2.42,13.88,13.23
2,2019-01-03,M01AB,2.90,3.89,14.32,11.43
3,2019-01-04,M01AB,3.39,5.11,15.42,12.03
4,2019-01-05,M01AB,3.48,4.58,12.68,9.20
5,2019-01-06,M01AB,3.28,5.38,13.86,10.58
6,2019-01-07,M01AB,0.92,2.93,11.89,10.98
7,2019-01-08,M01AB,2.82,3.13,13.32,10.50
8,2019-01-09,M01AB,2.87,4.11,12.79,9.92
9,2019-01-10,M01AB,2.88,5.26,14.25,11.37
